# 完整训练流程 - 从数据到索引

这个 notebook 完成：
1. 加载数据
2. 构建 FAISS 索引
3. 保存到 Google Drive
4. （可选）上传到 HuggingFace Hub

**运行环境**: Google Colab with GPU

## Step 0: Setup

In [ ]:
# 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 设置路径
import os
os.chdir('/content')

# 克隆代码仓库
if not os.path.exists('RAG-embedding'):
    !git clone https://github.com/Echo-Lee/RAG-embedding.git
else:
    print("✅ Repository already exists")
    !cd RAG-embedding && git pull

os.chdir('RAG-embedding')
print(f"✅ Current directory: {os.getcwd()}")

In [ ]:
# 安装依赖
!pip install -q sentence-transformers faiss-cpu pyyaml tqdm huggingface_hub

import sys
sys.path.insert(0, 'src')

print("✅ Dependencies installed")

## Step 1: 配置路径和参数

In [ ]:
from pathlib import Path

# ========== 配置 ==========

# Google Drive 路径
DRIVE_BASE = Path("/content/drive/MyDrive/Epiq Project/pipeline")

# 数据路径
DATA_PATHS = {
    'hospital': DRIVE_BASE / "data/processed/hospital/threads_with_summary.json",
    'corruption': DRIVE_BASE / "data/processed/corruption/emails_group_by_thread.json"
}

# 输出路径
OUTPUT_BASE = DRIVE_BASE / "faiss_index"
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

# 模型配置
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
BATCH_SIZE = 32

# 选择要处理的数据集
DATASETS_TO_BUILD = ['hospital', 'corruption']  # 或只选一个: ['hospital']

print("📁 Configuration:")
print(f"  Data base: {DRIVE_BASE}")
print(f"  Output base: {OUTPUT_BASE}")
print(f"  Embedding model: {EMBEDDING_MODEL}")
print(f"  Datasets: {DATASETS_TO_BUILD}")

## Step 2: 验证数据文件存在

In [ ]:
# 检查数据文件
print("🔍 Checking data files...\n")

for dataset, path in DATA_PATHS.items():
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"  ✅ {dataset}: {path}")
        print(f"     Size: {size_mb:.1f} MB")
    else:
        print(f"  ❌ {dataset}: NOT FOUND at {path}")
        print(f"     Please check the path!")

## Step 3: 加载数据

In [ ]:
from data.loader import DataLoader
from config.config import RAGConfig, DatasetConfig
import json

def load_dataset(dataset_name):
    """加载指定数据集"""
    print(f"\n📥 Loading {dataset_name} dataset...")

    # 创建配置
    dataset_config = DatasetConfig(
        name=dataset_name,
        data_path=DATA_PATHS[dataset_name],
        format='thread_format'
    )

    config = RAGConfig(
        dataset=dataset_config,
        embedding_model=EMBEDDING_MODEL,
        batch_size=BATCH_SIZE
    )

    # 加载数据
    loader = DataLoader(config)
    documents = loader.load()

    print(f"  ✅ Loaded {len(documents)} documents")
    print(f"  Sample document keys: {list(documents[0].keys())[:5]}")

    return documents, config

# 测试加载第一个数据集
test_dataset = DATASETS_TO_BUILD[0]
test_docs, test_config = load_dataset(test_dataset)

print(f"\n📊 Sample document:")
print(f"  Content preview: {test_docs[0]['content'][:200]}...")

## Step 4: 构建 FAISS 索引

In [ ]:
from retrieval.indexer import FAISSIndexer
import faiss
from datetime import datetime

def build_and_save_index(dataset_name, documents, config):
    """构建并保存 FAISS 索引"""
    print(f"\n{'='*60}")
    print(f"🔨 Building index for {dataset_name}")
    print(f"{'='*60}")

    # 创建索引器
    indexer = FAISSIndexer(config)

    # 构建索引（这会花费一些时间）
    print(f"\n⏳ Building FAISS index... (this may take 5-15 minutes)")
    index = indexer.build_index(documents)

    print(f"\n✅ Index built successfully!")
    print(f"  - Total vectors: {index.ntotal}")
    print(f"  - Dimension: {index.d}")

    # 保存索引
    output_dir = OUTPUT_BASE / dataset_name
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n💾 Saving index to: {output_dir}")

    # 保存 FAISS index
    index_path = output_dir / "faiss_index.bin"
    faiss.write_index(index, str(index_path))
    print(f"  ✅ Saved: {index_path}")

    # 保存元数据
    metadata = {
        'dataset_name': dataset_name,
        'num_docs': len(documents),
        'dimension': index.d,
        'index_type': 'IndexFlatIP',
        'embedding_model': config.embedding_model,
        'created_at': datetime.now().isoformat(),
        'description': f'FAISS index for {dataset_name} email dataset'
    }

    metadata_path = output_dir / "metadata.json"
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"  ✅ Saved: {metadata_path}")

    # 保存文档映射（用于后续检索）
    doc_metadata_path = output_dir / "documents.json"
    doc_metadata = [
        {
            'idx': i,
            'thread_id': doc.get('metadata', {}).get('thread_id', 'unknown'),
            'content_preview': doc['content'][:200]
        }
        for i, doc in enumerate(documents)
    ]
    with open(doc_metadata_path, 'w') as f:
        json.dump(doc_metadata, f, indent=2)
    print(f"  ✅ Saved: {doc_metadata_path}")

    print(f"\n🎉 {dataset_name} index complete!")
    print(f"{'='*60}\n")

    return output_dir

## Step 5: 批量构建所有索引

In [ ]:
import time

print("\n" + "="*60)
print("🚀 Starting batch index building")
print("="*60)

results = {}
start_time = time.time()

for dataset_name in DATASETS_TO_BUILD:
    try:
        # 加载数据
        documents, config = load_dataset(dataset_name)

        # 构建并保存索引
        output_dir = build_and_save_index(dataset_name, documents, config)

        results[dataset_name] = {
            'status': 'success',
            'num_docs': len(documents),
            'output_dir': str(output_dir)
        }

    except Exception as e:
        print(f"\n❌ Error processing {dataset_name}: {e}")
        import traceback
        traceback.print_exc()

        results[dataset_name] = {
            'status': 'failed',
            'error': str(e)
        }

total_time = time.time() - start_time

print("\n" + "="*60)
print("📊 Build Summary")
print("="*60)
print(f"\nTotal time: {total_time/60:.1f} minutes\n")

for dataset, result in results.items():
    if result['status'] == 'success':
        print(f"✅ {dataset}:")
        print(f"   - Documents: {result['num_docs']}")
        print(f"   - Output: {result['output_dir']}")
    else:
        print(f"❌ {dataset}: {result['error']}")

print("\n" + "="*60)

## Step 6: 验证索引

In [ ]:
# 验证索引文件
print("\n🔍 Verifying saved indexes...\n")

for dataset_name in DATASETS_TO_BUILD:
    output_dir = OUTPUT_BASE / dataset_name

    if output_dir.exists():
        print(f"📁 {dataset_name}:")

        # 检查文件
        index_file = output_dir / "faiss_index.bin"
        metadata_file = output_dir / "metadata.json"
        docs_file = output_dir / "documents.json"

        if index_file.exists():
            size_mb = index_file.stat().st_size / (1024 * 1024)
            print(f"  ✅ faiss_index.bin ({size_mb:.1f} MB)")

            # 尝试加载
            try:
                test_index = faiss.read_index(str(index_file))
                print(f"     - Vectors: {test_index.ntotal}")
                print(f"     - Dimension: {test_index.d}")
            except Exception as e:
                print(f"     ❌ Failed to load: {e}")
        else:
            print(f"  ❌ faiss_index.bin NOT FOUND")

        if metadata_file.exists():
            print(f"  ✅ metadata.json")
            with open(metadata_file) as f:
                meta = json.load(f)
                print(f"     - Created: {meta['created_at']}")
        else:
            print(f"  ❌ metadata.json NOT FOUND")

        if docs_file.exists():
            print(f"  ✅ documents.json")
        else:
            print(f"  ❌ documents.json NOT FOUND")

        print()
    else:
        print(f"❌ {dataset_name}: Directory not found")

## Step 7: (可选) 上传到 HuggingFace Hub

In [ ]:
# 如果要上传到 HF Hub，运行这个 cell

UPLOAD_TO_HF = False  # 改成 True 来启用上传
YOUR_HF_USERNAME = "YOUR_USERNAME"  # 改成你的 HF 用户名

if UPLOAD_TO_HF:
    if YOUR_HF_USERNAME == "YOUR_USERNAME":
        print("❌ Please set YOUR_HF_USERNAME first!")
    else:
        from huggingface_hub import login, HfApi

        # 登录
        print("📝 Login to HuggingFace Hub")
        print("Get your token from: https://huggingface.co/settings/tokens")
        hf_token = input("Enter HF Token: ")
        login(token=hf_token)

        # 创建 repo
        api = HfApi()
        repo_id = f"{YOUR_HF_USERNAME}/rag-indexes"

        print(f"\n📦 Creating repository: {repo_id}")
        api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)

        # 上传索引
        for dataset_name in DATASETS_TO_BUILD:
            output_dir = OUTPUT_BASE / dataset_name

            if output_dir.exists():
                print(f"\n📤 Uploading {dataset_name}...")
                api.upload_folder(
                    folder_path=str(output_dir),
                    repo_id=repo_id,
                    repo_type="dataset",
                    path_in_repo=dataset_name
                )
                print(f"  ✅ {dataset_name} uploaded")

        print(f"\n🎉 All uploaded!")
        print(f"View at: https://huggingface.co/datasets/{repo_id}")
else:
    print("ℹ️  Skipping HF Hub upload (UPLOAD_TO_HF = False)")
    print("   Indexes are saved to Google Drive")

## 完成！

### 你现在有：

✅ FAISS 索引文件在 Google Drive:  
   `/content/drive/MyDrive/Epiq Project/pipeline/faiss_index/`

### 下一步：

1. **如果上传到了 HF Hub**:  
   直接创建 Gradio Space，参考 `DEPLOY_GUIDE.md`

2. **如果没上传**:  
   先运行 `upload_to_hf.py` 上传索引，再创建 Space

### 测试索引（可选）

想测试索引是否工作？继续下面的 cell：

In [ ]:
# 测试检索
from sentence_transformers import SentenceTransformer
import numpy as np

# 选择数据集测试
test_dataset = DATASETS_TO_BUILD[0]
index_path = OUTPUT_BASE / test_dataset / "faiss_index.bin"

print(f"🧪 Testing retrieval on {test_dataset}...\n")

# 加载索引
index = faiss.read_index(str(index_path))
print(f"✅ Index loaded: {index.ntotal} vectors\n")

# 加载模型
model = SentenceTransformer(EMBEDDING_MODEL)
print(f"✅ Model loaded\n")

# 测试查询
test_query = "What are the main issues?"
print(f"🔍 Query: {test_query}\n")

# 编码查询
query_emb = model.encode([test_query], normalize_embeddings=True)

# 检索
scores, indices = index.search(query_emb, 5)

print(f"📋 Top 5 Results:\n")
for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
    print(f"{i+1}. Index: {idx}, Score: {score:.4f}")

print("\n✅ Index is working correctly!")